# Lumis 04b: Prior adjustment (how common each condition is)

Training used class weights, which make the model act as if all five conditions were **equally common**. That protects scabies, the smallest class, but it also makes the model over-predict scabies (only about 40% of its scabies predictions were right on validation). In Malaysia and Southeast Asia, eczema, fungal infections and acne are far more common than scabies.

**Prior adjustment** (also called logit adjustment) corrects for this after training, with no retraining: each class score is shifted by how common the condition is,

`adjusted score = model score + tau x log(prior)`

where `tau` sets the strength (0 = no adjustment, 1 = full correction to the prior).

**Prior used:** the class mix of the training data (eczema 31%, psoriasis 26%, tinea 23%, acne 13%, scabies 8%), as a proxy for a real-world case mix. Section 3 also checks a published Malaysian case mix (Hospital Kuala Lumpur, 2017) and suggests which prior to use.

This notebook tries several strengths on the **validation set** only and reports the trade-off. Pick one, then set the same `PRIOR_TAU` in notebook 04 and rerun it. CPU only; takes seconds.

## 1. Setup

In [ ]:
RUN = None             # None = the latest EfficientNet-B0 run
TAUS = [0.0, 0.25, 0.5, 0.75, 1.0]
CUSTOM_PRIOR = None    # e.g. {'acne': 0.2, 'eczema': 0.35, 'psoriasis': 0.1, 'scabies': 0.05, 'tinea': 0.3}
MIN_SCABIES_SENS = 0.50  # selection rule: best accuracy that keeps scabies sensitivity at or above this

from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import confusion_matrix, f1_score

BASE = '/content/drive/MyDrive/Lumis'
runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN is None:
    RUN = runs[runs.backbone == 'efficientnetb0'].iloc[-1]['run']
info = runs[runs.run == RUN].iloc[-1]
split_name = info.get('split', 'v1') if isinstance(info.get('split', 'v1'), str) else 'v1'
split_all = pd.read_csv(f'{BASE}/data/lumis_split_{split_name}.csv')
CLASSES = sorted(split_all.lumis_class.unique())
K = len(CLASSES)

logits = np.load(f'{BASE}/exports/runs/{RUN}/val_logits.npy')
y = np.load(f'{BASE}/exports/runs/{RUN}/val_labels.npy')

if CUSTOM_PRIOR:
    prior = np.array([CUSTOM_PRIOR[c] for c in CLASSES], dtype=float)
    prior = prior / prior.sum()
    prior_source = 'custom'
else:
    counts = split_all[split_all.split == 'train'].lumis_class.value_counts()
    prior = np.array([counts[c] for c in CLASSES]) / counts.sum()
    prior_source = 'training class mix'
print(RUN, '| prior from', prior_source)
print(dict(zip(CLASSES, np.round(prior, 3))))

## 2. Compare adjustment strengths on validation

In [ ]:
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(z, y):
    nll = lambda lt: -np.mean(np.log(softmax(z / np.exp(lt))[np.arange(len(y)), y] + 1e-12))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method='bounded').x))

S = CLASSES.index('scabies')
rows = []
for tau in TAUS:
    adj = tau * (np.log(prior) - np.log(prior).mean())
    z = logits + adj
    T = fit_temperature(z, y)
    pred = softmax(z / T).argmax(1)
    cm = confusion_matrix(y, pred, labels=range(K))
    sens = cm.diagonal() / cm.sum(1)
    row = {'tau': tau, 'accuracy': (pred == y).mean(),
           'macro_f1': f1_score(y, pred, average='macro', zero_division=0),
           'macro_sensitivity': sens.mean(),
           'scabies_precision': cm[S, S] / cm[:, S].sum() if cm[:, S].sum() else np.nan,
           'share_predicted_scabies': (pred == S).mean(), 'temperature': T}
    row.update({f'sens_{c}': sens[i] for i, c in enumerate(CLASSES)})
    rows.append(row)
cmp = pd.DataFrame(rows).set_index('tau').round(3)
display(cmp)
cmp.to_csv(f'{BASE}/exports/runs/{RUN}/prior_adjustment_comparison.csv')

ok = cmp[cmp.sens_scabies >= MIN_SCABIES_SENS]
best = ok.accuracy.idxmax() if len(ok) else 0.0
print(f'Suggested PRIOR_TAU = {best}: highest validation accuracy with scabies sensitivity >= {MIN_SCABIES_SENS:.0%}')
print('Set PRIOR_TAU to this value in notebook 04 and rerun it.')

## 3. Check against the Malaysian case mix

Validation is in DermNet's mix of conditions, which is not what a Malaysian clinic sees. The only published Malaysian clinic-wide figures found are from the **Hospital Kuala Lumpur dermatology clinic, 2008-2014** (Heah, Mohd Noor & Johar, *Malaysian Journal of Dermatology* 2017, 38:19-25): eczema/dermatitis 39.1%, acne vulgaris 9.6%, psoriasis 9.5%, tinea + pityriasis versicolor 8.7%. Scabies is **not reported** in that paper, so its share is a stated assumption (5%), re-checked at 2% and 8%.

This section re-weights each validation result to that mix (expected accuracy = sum over classes of mix share x sensitivity) and also tries using the Malaysian mix itself as the adjustment prior. Caveat: HKL is a tertiary referral centre, so psoriasis is likely over-represented and tinea and scabies under-represented compared with what the public would photograph.

In [ ]:
# Published Malaysian case mix: Heah, Mohd Noor & Johar (2017), "Prevalence of Skin Diseases in
# Dermatology Outpatient Clinic, Hospital Kuala Lumpur", Malaysian Journal of Dermatology 38:19-25.
# 2008-2014, about 58,000 patients. Numbers are % of all clinic diagnoses.
HKL = {'eczema': 39.07,    # all eczema/dermatitis
       'acne': 9.58,       # acne vulgaris
       'psoriasis': 9.54,  # all psoriasis
       'tinea': 8.72}      # tinea corporis 3.78 + cruris 1.96 + pedis 1.49 + capitis 0.64 + pityriasis versicolor 0.85
SCABIES_SHARE = 0.05       # NOT in the HKL paper: a stated assumption, re-checked at 2% and 8% below

def malaysian_mix(scabies_share):
    m = np.array([HKL.get(c, 0.0) for c in CLASSES], dtype=float)
    m = m / m.sum() * (1 - scabies_share)
    m[S] = scabies_share
    return m

def field_metrics(cm, mix):
    # Re-weight the validation results as if patients arrived in the Malaysian mix
    p_pred = cm / cm.sum(1, keepdims=True)          # row = true class, column = predicted class
    joint = mix[:, None] * p_pred
    share = joint.sum(0)
    prec = np.divide(joint.diagonal(), share, out=np.full(K, np.nan), where=share > 0)
    return np.trace(joint), prec, share

def compare(mix):
    rows = []
    for prior_name, pr in [('training mix', prior), ('Malaysian mix', mix)]:
        for tau in TAUS:
            if prior_name == 'Malaysian mix' and tau == 0:
                continue                             # same as training mix at tau 0
            z = logits + tau * (np.log(pr) - np.log(pr).mean())
            T = fit_temperature(z, y)
            pred = softmax(z / T).argmax(1)
            cm = confusion_matrix(y, pred, labels=range(K))
            sens = cm.diagonal() / cm.sum(1)
            acc_my, prec_my, share_my = field_metrics(cm, mix)
            rows.append({'prior': prior_name, 'tau': tau,
                         'val_accuracy': (pred == y).mean(),
                         'MY_expected_accuracy': acc_my,
                         'MY_scabies_precision': prec_my[S],
                         'MY_share_told_scabies': share_my[S],
                         **{f'sens_{c}': sens[i] for i, c in enumerate(CLASSES)}})
    return pd.DataFrame(rows).set_index(['prior', 'tau']).round(3)

def pick(tab):
    ok = tab[tab.sens_scabies >= MIN_SCABIES_SENS]
    return tuple(ok.MY_expected_accuracy.idxmax()) if len(ok) else ('training mix', 0.0)

my_mix = malaysian_mix(SCABIES_SHARE)
print('Malaysian mix used:', {c: round(float(v), 3) for c, v in zip(CLASSES, my_mix)})
print('Training mix:      ', {c: round(float(v), 3) for c, v in zip(CLASSES, prior)})
my_tab = compare(my_mix)
display(my_tab)
my_tab.to_csv(f'{BASE}/exports/runs/{RUN}/prior_adjustment_malaysian_mix.csv')

best_prior, best_tau = pick(my_tab)
print(f'\nSuggested: prior = {best_prior}, PRIOR_TAU = {best_tau}')
print(f'  (highest expected accuracy on the Malaysian mix with scabies sensitivity >= {MIN_SCABIES_SENS:.0%})')
for alt in [0.02, 0.08]:
    p, t = pick(compare(malaysian_mix(alt)))
    print(f'  If scabies were {alt:.0%} of cases instead of {SCABIES_SHARE:.0%}: pick would be {p}, tau {float(t)}')
if best_prior == 'Malaysian mix':
    print('\nPaste into notebook 04:')
    print(f'PRIOR_TAU = {float(best_tau)}')
    print('CUSTOM_PRIOR =', {c: round(float(v), 4) for c, v in zip(CLASSES, my_mix)})
else:
    print(f'\nIn notebook 04: PRIOR_TAU = {best_tau}, CUSTOM_PRIOR = None')